<a href="https://colab.research.google.com/github/fadeeva/MLDL_plgrnd/blob/master/CV/courses_notes/stepik__object_detection/2_nms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

In [4]:
def iou(box1, box2):
    '''
        Ограничевающие рамки в формате (x1, y1, x2, y2).

        Parameters:
            box1: torch.Tensor, size(4, )
            box2: torch.Tensor, size(4, )
        Returns:
            iou: torch.Tensor (scalar)
    '''

    b1x1, b1y1, b1x2, b1y2 = box1
    b2x1, b2y1, b2x2, b2y2 = box2

    area1 = (b1x2 - b1x1)*(b1y2 - b1y1)
    area2 = (b2x2 - b2x1)*(b2y2 - b2y1)

    x_left = torch.max(b1x1, b2x1)
    y_top = torch.max(b1y1, b2y1)
    x_right = torch.min(b1x2, b2x2)
    y_bottom = torch.min(b1y2, b2y2)

    if x_right < x_left or y_bottom < y_top:
        return torch.tensor(0, dtype=torch.float)

    w = x_right - x_left
    h = y_bottom - y_top

    inter = w*h
    union = area1 + area2 - inter
    iou = inter / union

    return iou

In [6]:
def box_iou(boxes1, boxes2):
    '''
        Ограничевающие рамки в формате (x1, y1, x2, y2)

        Parameters:
            boxes1: torch.Tensor, size(N, 4)
            boxes2: torch.Tensor, size(M, 4)
        Returns:
            iou: torch.Tensor, size(N, M)
    '''

    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])

    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2]) # size (N, M, 2)
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:]) # size (N, M, 2)

    wh = (rb - lt).clamp(min=0) # size (N, M, 2)

    inter = wh[..., 0] * wh[..., 1] # size (N, M)
    union = area1[:, None] + area2 - inter # size (N, M)

    iou = inter / union  # size (N, M)

    return iou


In [5]:
def xywh2xyxy(inp):
    '''
        Преобразование формата ограниченных рамок. (x, y, w, h) -> (x, y, x, y)
        Parameters:
            inp: torch.Tensor, size(batch_size, num_boxes, 4) or (num_boxes, 4).
                Формат ограниченных рамок (x, y, w, h).
        Returns:
            out: torch.Tensor, size(batch_size, num_boxes, 4) or (num_boxes, 4).
                Формат ограниченных рамок (x, y, x, y).
    '''

    out = torch.empty_like(inp)

    xy = inp[..., :2] # координаты x и y центра ограничевающей рамки
    wh = inp[..., 2:] / 2 # половина высоты и ширины ограничивающей рамки

    out[..., :2] = xy - wh # координаты x и y левого верхнего угла
    out[..., 2:] = xy + wh # координаты x и y правого нижнего угла

    return out


In [3]:
def xyxy2xywh(inp):
    '''
        Преобразование формата ограниченных рамок. (x, y, x, y) -> (x, y, w, h)
        Parameters:
            inp: torch.Tensor, size(batch_size, num_boxes, 4) or (num_boxes, 4).
                Формат ограниченных рамок (x, y, x, y).
        Returns:
            out: torch.Tensor, size(batch_size, num_boxes, 4) or (num_boxes, 4).
                Формат ограниченных рамок (x, y, w, h).
    '''

    out = torch.empty_like(inp)

    # координаты центра ограничивающей рамки
    out[..., 0] = (inp[..., 0] + inp[..., 2]) / 2 # x
    out[..., 1] = (inp[..., 1] + inp[..., 3]) / 2 # y

    out[..., 2] = inp[..., 2] - inp[..., 0] # ширина рамки
    out[..., 3] = inp[..., 3] - inp[..., 1] # высота рамки

    return out


In [7]:
def nms(boxes, scores, threshold=.5):
    _, sorted_idx = scores.sort(descending=True)

    keep = []
    while sorted_idx.numel() > 0:
        if sorted_idx.numel() == 1:
            keep.append(sorted_idx)
            break

        idx = sorted_idx[0]
        keep.append(idx)

        boxs1 = boxes[sorted_idx[0]].unsqueeze(dim=0) # size (1, 4)
        boxs2 = boxes[sorted_idx[1:]]                 # size (M, 4)
        iou = box_iou(boxs1, boxs2)

        i = (iou < threshold).nonzero()[:, 1]
        if i.numel()==0:
            break

        sorted_idx = sorted_idx[i+1]

    return torch.tensor(keep, dtype=torch.int)
